In [19]:
#%% [0] 基本依赖（如需）
# 如果缺少 openpyxl / joblib / torch 等，可取消下面注释安装
# %pip install pandas numpy scikit-learn openpyxl joblib torch --quiet

# 你的数据与输出目录
EXCEL_PATH = "updated_fc_predictions.xlsx"  # 改成你的绝对/相对路径
SHEET_NAME = "Sheet1"
OUTDIR = "experiment_outputs"  # 输出目录


In [20]:
#%% [1] Imports & config
import os, json, math
from dataclasses import dataclass, asdict
from typing import List, Dict, Any
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.ensemble import RandomForestRegressor
import joblib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 固定初始训练/测试划分：85/15（≈8:2）
TEST_SIZE = 0.15
SPLIT_SEED = 42

# 十档训练子集比例（相对于训练集）
TRAIN_FRACS = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]

# 每档五个随机种子
SEEDS = [42, 43, 44, 45, 46]

# 参与实验的模型（可自由增删）
MODELS = ["torch_mlp", "torch_ann", "svr", "knn", "bayesian_ridge", "random_forest"]


# 特征与目标列
FEATURES = ['AGE', 'PC', 'PC_TYPE', 'FA', 'SS', 'SF', 'FAGG', 'CAGG', 'WATER', 'AEA', 'WR_HR', 'WR', 'ACC', 
        'VOID', 'w/b', 'b/a', 'CAGG%', 'FAGG%', 'FA%', 'SS%', 'SF%']
TARGET_COL = 'fc (MPa)'  # 确保表中存在该列

# 设备选择
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 输出目录结构
MODELS_DIR = os.path.join(OUTDIR, "models")
RESULTS_DIR = os.path.join(OUTDIR, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [21]:
#%% [2] Data loading & scaling
def load_data(excel_path: str, sheet_name: str, features: List[str], target_col: str):
    df = pd.read_excel(excel_path, sheet_name=sheet_name).copy()
    df = df.dropna(subset=features + [target_col]).reset_index(drop=True)
    X = df[features].values
    y = df[target_col].values.astype(float)
    return X, y, df

def make_split_and_scalers(X, y, test_size: float, split_seed: int):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=split_seed
    )
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()

    X_train_n = x_scaler.fit_transform(X_train)
    X_test_n  = x_scaler.transform(X_test)

    y_train_n = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
    y_test_n  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

    return X_train, X_test, y_train, y_test, X_train_n, X_test_n, y_train_n, y_test_n, x_scaler, y_scaler

X, y, _df = load_data(EXCEL_PATH, SHEET_NAME, FEATURES, TARGET_COL)
Xtr, Xte, ytr, yte, Xtr_n, Xte_n, ytr_n, yte_n, x_scaler, y_scaler = make_split_and_scalers(
    X, y, TEST_SIZE, SPLIT_SEED
)

print(f"Train size: {len(Xtr)} | Test size: {len(Xte)} | Features: {Xtr.shape[1]}")


Train size: 5740 | Test size: 1013 | Features: 21


In [22]:
#%% [3] Torch model & custom loss
@dataclass
class TorchHyperParams:
    layers: int = 2
    neurons: int = 52
    lr: float = 7.6e-4
    epochs: int = 300
    batch_size: int = 8
    a: float = 40.5
    b: float = 15.3
    e: float = 6.5
    d: float = 0.36

HPS = TorchHyperParams()

class RegressionModel(nn.Module):
    def __init__(self, input_dim: int, layers: int, neurons: int):
        super().__init__()
        seq = [nn.Linear(input_dim, neurons), nn.ReLU()]
        for _ in range(layers - 1):
            seq += [nn.Linear(neurons, neurons), nn.ReLU()]
        seq += [nn.Linear(neurons, 1)]
        self.net = nn.Sequential(*seq)

    def forward(self, x):
        return self.net(x).squeeze(-1)

def custom_loss(outputs, targets, inputs, a, b, e, d, wb_index):
    mse_loss = nn.MSELoss()(outputs, targets)
#     AGE = torch.tensor(7.0, device=outputs.device, dtype=outputs.dtype)
    AGE = inputs[:, 0]
#     wb = inputs[:, wb_index]  # 使用标准化后的 w/b 特征
    wb = inputs[:, -7]
    fc_pred = (a * torch.log(AGE) + b) * (e * torch.pow(AGE, d)) ** (-wb)
    residual = torch.abs(outputs - fc_pred)
    residual = torch.nan_to_num(residual, nan=0.0, posinf=1e10, neginf=-1e10)
    mean_square_residual = torch.mean(residual ** 2)
    residual_norm = residual * torch.sqrt(mse_loss / mean_square_residual) if mean_square_residual.item() > 0 else residual
    return 0.5 * mse_loss + 0.5 * torch.mean(residual_norm)

def train_torch_mlp(Xn_train, yn_train, frac: float, seed: int, hps: TorchHyperParams, wb_index: int, device: str):
    rng = np.random.default_rng(seed)
    n = len(Xn_train)
    m = max(1, int(round(frac * n)))
    idx = rng.choice(n, size=m, replace=False)

    X_sub = torch.tensor(Xn_train[idx], dtype=torch.float32)
    y_sub = torch.tensor(yn_train[idx], dtype=torch.float32)
    ds = TensorDataset(X_sub, y_sub)
    dl = DataLoader(ds, batch_size=hps.batch_size, shuffle=True)

    model = RegressionModel(input_dim=Xn_train.shape[1], layers=hps.layers, neurons=hps.neurons).to(device)
    opt = optim.Adam(model.parameters(), lr=hps.lr)

    for _ in range(hps.epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = custom_loss(pred, yb, xb, hps.a, hps.b, hps.e, hps.d, wb_index)
            loss.backward()
            opt.step()
    return model


In [23]:
#%% [4] Optional sklearn trainers
def _subidx(n, frac, seed):
    rng = np.random.default_rng(seed)
    m = max(1, int(round(frac * n)))
    return rng.choice(n, size=m, replace=False)

#%% [3b] Plain ANN (same architecture, standard MSE loss)

def train_torch_ann(Xn_train, yn_train, frac: float, seed: int, hps: TorchHyperParams, device: str):
    rng = np.random.default_rng(seed)
    n = len(Xn_train)
    m = max(1, int(round(frac * n)))
    idx = rng.choice(n, size=m, replace=False)

    X_sub = torch.tensor(Xn_train[idx], dtype=torch.float32)
    y_sub = torch.tensor(yn_train[idx], dtype=torch.float32)
    ds = TensorDataset(X_sub, y_sub)
    dl = DataLoader(ds, batch_size=hps.batch_size, shuffle=True)

    model = RegressionModel(input_dim=Xn_train.shape[1], layers=hps.layers, neurons=hps.neurons).to(device)
    opt = optim.Adam(model.parameters(), lr=hps.lr)
    criterion = nn.MSELoss()

    for _ in range(hps.epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)   # ← 纯 MSELoss
            loss.backward()
            opt.step()
    return model


def train_svr(Xn_train, yn_train, frac: float, seed: int):
    idx = _subidx(len(Xn_train), frac, seed)
    model = SVR(kernel='rbf', C=10.0, epsilon=0.1, gamma='scale')
    model.fit(Xn_train[idx], yn_train[idx])
    return model

def train_knn(Xn_train, yn_train, frac: float, seed: int):
    idx = _subidx(len(Xn_train), frac, seed)
    model = KNeighborsRegressor(n_neighbors=5, weights='distance')
    model.fit(Xn_train[idx], yn_train[idx])
    return model

def train_bayes_ridge(Xn_train, yn_train, frac: float, seed: int):
    idx = _subidx(len(Xn_train), frac, seed)
    model = BayesianRidge()
    model.fit(Xn_train[idx], yn_train[idx])
    return model

def train_random_forest(Xn_train, yn_train, frac: float, seed: int):
    idx = _subidx(len(Xn_train), frac, seed)
    model = RandomForestRegressor(n_estimators=400, random_state=seed, n_jobs=-1)
    model.fit(Xn_train[idx], yn_train[idx])
    return model


In [24]:
#%% [5] Evaluation on original scale (MPa)
def evaluate_on_test(model, model_type: str, Xn_test, y_test, y_scaler: StandardScaler, device: str):
#     if model_type == "torch_mlp":
    if model_type in ("torch_mlp", "torch_ann"):
        model.eval()
        with torch.no_grad():
            X_t = torch.tensor(Xn_test, dtype=torch.float32, device=device)
            y_pred_n = model(X_t).cpu().numpy().ravel()
    else:
        y_pred_n = model.predict(Xn_test).ravel()

    y_pred = y_scaler.inverse_transform(y_pred_n.reshape(-1, 1)).ravel()
    r2 = r2_score(y_test, y_pred)
    rmse = math.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    return {"R2": r2, "RMSE": rmse, "MAE": mae}, y_pred


In [25]:
#%% [6] Experiment loop
wb_index = FEATURES.index('w/b')  # 自定义损失需要

all_rows = []

for model_type in MODELS:
    print(f"\n=== Model: {model_type} ===")
    for frac in TRAIN_FRACS:
        for seed in SEEDS:
            run_id = f"{model_type}_p{int(frac*100)}_seed{seed}"
            run_dir = os.path.join(MODELS_DIR, run_id)
            os.makedirs(run_dir, exist_ok=True)

            # 训练
            if model_type == "torch_mlp":
                model = train_torch_mlp(Xtr_n, ytr_n, frac, seed, HPS, wb_index, DEVICE)
                torch.save(model.state_dict(), os.path.join(run_dir, "KINN_model.pt"))
            elif model_type == "svr":
                model = train_svr(Xtr_n, ytr_n, frac, seed)
                joblib.dump(model, os.path.join(run_dir, "SVR_model.joblib"))
            elif model_type == "knn":
                model = train_knn(Xtr_n, ytr_n, frac, seed)
                joblib.dump(model, os.path.join(run_dir, "KNN_model.joblib"))
            elif model_type == "bayesian_ridge":
                model = train_bayes_ridge(Xtr_n, ytr_n, frac, seed)
                joblib.dump(model, os.path.join(run_dir, "BRmodel.joblib"))
            elif model_type == "random_forest":
                model = train_random_forest(Xtr_n, ytr_n, frac, seed)
                joblib.dump(model, os.path.join(run_dir, "RFmodel.joblib"))
            elif model_type == "torch_ann":
                model = train_torch_ann(Xtr_n, ytr_n, frac, seed, HPS, DEVICE)
                torch.save(model.state_dict(), os.path.join(run_dir, "ANNmodel.pt"))
            else:
                raise ValueError(f"Unknown model_type: {model_type}")

            # 保存每次运行的配置与 scaler（便于复现实验）
            per_run_cfg = {
                "model_type": model_type,
                "train_fraction": frac,
                "seed": seed,
                "features": FEATURES,
                "target_col": TARGET_COL,
                "split_seed": SPLIT_SEED,
                "torch_hparams": asdict(HPS) if model_type == 'torch_mlp' else None
            }
            with open(os.path.join(run_dir, "run_config.json"), "w") as f:
                json.dump(per_run_cfg, f, indent=2)

            joblib.dump(x_scaler, os.path.join(run_dir, "x_scaler.joblib"))
            joblib.dump(y_scaler, os.path.join(run_dir, "y_scaler.joblib"))

            # 评估（在不变的测试集，并将预测反变换回 MPa）
            metrics, y_pred = evaluate_on_test(model, model_type, Xte_n, yte, y_scaler, DEVICE)

            row = {
                "model": model_type,
                "train_fraction": frac,
                "seed": seed,
                "R2": metrics["R2"],
                "RMSE": metrics["RMSE"],
                "MAE": metrics["MAE"],
                "n_train_total": len(Xtr_n),
                "n_train_used": max(1, int(round(frac * len(Xtr_n)))),
                "n_test": len(Xte_n),
            }
            all_rows.append(row)

            print(f"{run_id:>30} | R2={row['R2']:.4f}, RMSE={row['RMSE']:.3f}, MAE={row['MAE']:.3f}")

# 汇总结果到 Excel
results_df = pd.DataFrame(all_rows)
excel_out = os.path.join(RESULTS_DIR, "metrics_summary.xlsx")
with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    results_df.to_excel(writer, index=False, sheet_name="metrics")

print("\nDone.")
print("Metrics saved to:", excel_out)
print("Models saved under:", MODELS_DIR)
results_df.head()



=== Model: torch_mlp ===
         torch_mlp_p100_seed42 | R2=0.7031, RMSE=7.505, MAE=5.458
         torch_mlp_p100_seed43 | R2=0.7117, RMSE=7.396, MAE=5.380
         torch_mlp_p100_seed44 | R2=0.7184, RMSE=7.309, MAE=5.331
         torch_mlp_p100_seed45 | R2=0.7111, RMSE=7.403, MAE=5.395
         torch_mlp_p100_seed46 | R2=0.6993, RMSE=7.552, MAE=5.529
          torch_mlp_p90_seed42 | R2=0.7015, RMSE=7.526, MAE=5.511
          torch_mlp_p90_seed43 | R2=0.6909, RMSE=7.657, MAE=5.679
          torch_mlp_p90_seed44 | R2=0.6939, RMSE=7.621, MAE=5.498
          torch_mlp_p90_seed45 | R2=0.6946, RMSE=7.612, MAE=5.463
          torch_mlp_p90_seed46 | R2=0.6372, RMSE=8.296, MAE=6.044
          torch_mlp_p80_seed42 | R2=0.6738, RMSE=7.867, MAE=5.715
          torch_mlp_p80_seed43 | R2=0.6869, RMSE=7.708, MAE=5.633
          torch_mlp_p80_seed44 | R2=0.6547, RMSE=8.094, MAE=5.844
          torch_mlp_p80_seed45 | R2=0.6903, RMSE=7.665, MAE=5.636
          torch_mlp_p80_seed46 | R2=0.6669, RMSE=7

                svr_p60_seed45 | R2=0.6792, RMSE=7.801, MAE=5.530
                svr_p60_seed46 | R2=0.7009, RMSE=7.533, MAE=5.429
                svr_p50_seed42 | R2=0.6724, RMSE=7.883, MAE=5.625
                svr_p50_seed43 | R2=0.6736, RMSE=7.869, MAE=5.695
                svr_p50_seed44 | R2=0.6790, RMSE=7.804, MAE=5.600
                svr_p50_seed45 | R2=0.6691, RMSE=7.923, MAE=5.619
                svr_p50_seed46 | R2=0.6678, RMSE=7.938, MAE=5.688
                svr_p40_seed42 | R2=0.6380, RMSE=8.287, MAE=5.913
                svr_p40_seed43 | R2=0.6444, RMSE=8.214, MAE=5.885
                svr_p40_seed44 | R2=0.6585, RMSE=8.048, MAE=5.767
                svr_p40_seed45 | R2=0.6527, RMSE=8.117, MAE=5.823
                svr_p40_seed46 | R2=0.6457, RMSE=8.198, MAE=5.891
                svr_p30_seed42 | R2=0.6504, RMSE=8.143, MAE=5.879
                svr_p30_seed43 | R2=0.6434, RMSE=8.225, MAE=5.985
                svr_p30_seed44 | R2=0.6499, RMSE=8.149, MAE=5.940
          

     random_forest_p100_seed42 | R2=0.7185, RMSE=7.308, MAE=5.200
     random_forest_p100_seed43 | R2=0.7158, RMSE=7.343, MAE=5.224
     random_forest_p100_seed44 | R2=0.7178, RMSE=7.316, MAE=5.205
     random_forest_p100_seed45 | R2=0.7187, RMSE=7.305, MAE=5.201
     random_forest_p100_seed46 | R2=0.7179, RMSE=7.315, MAE=5.219
      random_forest_p90_seed42 | R2=0.7129, RMSE=7.380, MAE=5.304
      random_forest_p90_seed43 | R2=0.7129, RMSE=7.380, MAE=5.262
      random_forest_p90_seed44 | R2=0.7162, RMSE=7.338, MAE=5.236
      random_forest_p90_seed45 | R2=0.7058, RMSE=7.471, MAE=5.286
      random_forest_p90_seed46 | R2=0.7175, RMSE=7.321, MAE=5.207
      random_forest_p80_seed42 | R2=0.7048, RMSE=7.483, MAE=5.374
      random_forest_p80_seed43 | R2=0.7174, RMSE=7.322, MAE=5.276
      random_forest_p80_seed44 | R2=0.7064, RMSE=7.463, MAE=5.342
      random_forest_p80_seed45 | R2=0.7148, RMSE=7.356, MAE=5.289
      random_forest_p80_seed46 | R2=0.7060, RMSE=7.469, MAE=5.318
      rand

,model,train_fraction,seed,R2,RMSE,MAE,n_train_total,n_train_used,n_test
0,torch_mlp,1.0,42,0.703077,7.505308,5.457750,5740,5740,1013
1,torch_mlp,1.0,43,0.711700,7.395525,5.379616,5740,5740,1013
2,torch_mlp,1.0,44,0.718389,7.309226,5.330587,5740,5740,1013
3,torch_mlp,1.0,45,0.711089,7.403352,5.395121,5740,5740,1013
4,torch_mlp,1.0,46,0.699347,7.552305,5.528808,5740,5740,1013


In [26]:
results_df

,model,train_fraction,seed,R2,RMSE,MAE,n_train_total,n_train_used,n_test
0,torch_mlp,1.0,42,0.703077,7.505308,5.457750,5740,5740,1013
1,torch_mlp,1.0,43,0.711700,7.395525,5.379616,5740,5740,1013
2,torch_mlp,1.0,44,0.718389,7.309226,5.330587,5740,5740,1013
3,torch_mlp,1.0,45,0.711089,7.403352,5.395121,5740,5740,1013
4,torch_mlp,1.0,46,0.699347,7.552305,5.528808,5740,5740,1013
...,...,...,...,...,...,...,...,...,...
295,random_forest,0.1,42,0.587452,8.846751,6.549269,5740,574,1013
296,random_forest,0.1,43,0.602391,8.685097,6.338399,5740,574,1013
297,random_forest,0.1,44,0.616923,8.524906,6.391017,5740,574,1013
298,random_forest,0.1,45,0.571917,9.011787,6.613011,5740,574,1013
